In [3]:
import pandas as pd
import os

# 1️⃣ 경로 설정
data_path = os.path.join("..", "data", "gics")
output_path = os.path.join("..", "output")

# output 폴더가 없으면 자동 생성
os.makedirs(output_path, exist_ok=True)

# 2️⃣ CSV 파일 불러오기
sector = pd.read_csv(os.path.join(data_path, "sector.csv"))
industry_group = pd.read_csv(os.path.join(data_path, "industry_group.csv"))
industry = pd.read_csv(os.path.join(data_path, "industry.csv"))
sub_industry = pd.read_csv(os.path.join(data_path, "sub_industry.csv"))

# 📊 데이터 로드 후 기본 정보 확인
print("=" * 60)
print("📁 원본 데이터 확인")
print("=" * 60)
print(f"sector.csv: {len(sector)}개 행")
print(f"industry_group.csv: {len(industry_group)}개 행")
print(f"industry.csv: {len(industry)}개 행")
print(f"sub_industry.csv: {len(sub_industry)}개 행")
print()

# 3️⃣ 공통 키 통일
for df in [sector, industry_group, industry, sub_industry]:
    if 'Symbol' in df.columns:
        df.rename(columns={'Symbol': 'symbol'}, inplace=True)

# 4️⃣ 병합 (symbol 기준 outer join)
merged = (
    sector
    .merge(industry_group, on='symbol', how='outer', suffixes=('_sector', '_indgrp'))
    .merge(industry, on='symbol', how='outer', suffixes=('', '_ind'))
    .merge(sub_industry, on='symbol', how='outer', suffixes=('', '_subind'))
)

# 5️⃣ 컬럼 이름 통일 (symbol, sector, industry_group, industry, sub_industry)
merged.columns = ['symbol', 'sector', 'industry_group', 'industry', 'sub_industry']

# 🔍 결측치 및 이상한 값 확인
print("=" * 60)
print("🔍 결측치 확인")
print("=" * 60)
missing_counts = merged.isnull().sum()
missing_pct = (merged.isnull().sum() / len(merged) * 100).round(2)
missing_df = pd.DataFrame({
    '결측치 개수': missing_counts,
    '결측치 비율(%)': missing_pct
})
print(missing_df)
print()

# symbol이 결측인 행 확인
if merged['symbol'].isnull().any():
    print("⚠️ symbol이 결측인 행이 있습니다:")
    print(merged[merged['symbol'].isnull()])
    print()

# 중복 symbol 확인
duplicates = merged[merged.duplicated(subset=['symbol'], keep=False)]
if len(duplicates) > 0:
    print("=" * 60)
    print("⚠️ 중복된 symbol 발견")
    print("=" * 60)
    print(duplicates.sort_values('symbol'))
    print()

# 모든 GICS 컬럼이 결측인 행 확인
all_gics_null = merged[
    merged['sector'].isnull() & 
    merged['industry_group'].isnull() & 
    merged['industry'].isnull() & 
    merged['sub_industry'].isnull()
]
if len(all_gics_null) > 0:
    print("=" * 60)
    print("⚠️ GICS 정보가 없는 symbol")
    print("=" * 60)
    print(all_gics_null['symbol'].tolist())
    print()

# 부분적으로 결측인 행 확인 (일부만 결측)
partial_null = merged[
    (merged['sector'].isnull() | 
     merged['industry_group'].isnull() | 
     merged['industry'].isnull() | 
     merged['sub_industry'].isnull()) &
    ~(merged['sector'].isnull() & 
      merged['industry_group'].isnull() & 
      merged['industry'].isnull() & 
      merged['sub_industry'].isnull())
]
if len(partial_null) > 0:
    print("=" * 60)
    print("⚠️ 부분적으로 GICS 정보가 없는 symbol")
    print("=" * 60)
    print(partial_null)
    print()

# 데이터 요약
print("=" * 60)
print("📊 데이터 요약")
print("=" * 60)
print(f"전체 행 수: {len(merged)}")
print(f"고유 symbol 수: {merged['symbol'].nunique()}")
print(f"완전한 데이터(결측 없음): {len(merged.dropna())}개")
print(f"GICS 정보 완전 결측: {len(all_gics_null)}개")
print(f"GICS 정보 부분 결측: {len(partial_null)}개")
print()

# 각 컬럼별 고유값 개수
print("각 컬럼별 고유값 개수:")
for col in ['sector', 'industry_group', 'industry', 'sub_industry']:
    unique_count = merged[col].nunique()
    print(f"  {col}: {unique_count}개")
print()

# 6️⃣ output 폴더에 2개 버전 저장
print("=" * 60)
print("💾 파일 저장")
print("=" * 60)

# 버전 1: 결측치 포함 (전체 데이터)
output_file_all = os.path.join(output_path, "gics_all.csv")
merged.to_csv(output_file_all, index=False, encoding='utf-8-sig')
print(f"✅ [전체 데이터] {len(merged)}개 행 저장")
print(f"   📂 {output_file_all}")

# 버전 2: 결측치 제거 (완전한 데이터만)
merged_clean = merged.dropna()
output_file_clean = os.path.join(output_path, "gic_clean.csv")
merged_clean.to_csv(output_file_clean, index=False, encoding='utf-8-sig')
print(f"✅ [결측치 제거] {len(merged_clean)}개 행 저장")
print(f"   📂 {output_file_clean}")

print()
print("=" * 60)
print("💡 사용 가이드")
print("=" * 60)
print("• gics_all.csv (349개)")
print("  → 가격 데이터만으로 예측할 때 사용")
print("  → 모든 symbol 포함")
print()
print("• gics_clean.csv (339개)")
print("  → GICS 정보를 feature로 사용할 때")
print("  → 완전한 GICS 정보만 포함")
print("=" * 60)

📁 원본 데이터 확인
sector.csv: 349개 행
industry_group.csv: 349개 행
industry.csv: 349개 행
sub_industry.csv: 349개 행

🔍 결측치 확인
                결측치 개수  결측치 비율(%)
symbol               0       0.00
sector              10       2.87
industry_group      10       2.87
industry            10       2.87
sub_industry        10       2.87

⚠️ GICS 정보가 없는 symbol
['ANKPII', 'FXIKLD', 'HZBYDG', 'IEPPEH', 'KWLQBG', 'MCUUGN', 'TVTYVW', 'XBTGHA', 'YDSAWP', 'YGUFCU']

📊 데이터 요약
전체 행 수: 349
고유 symbol 수: 349
완전한 데이터(결측 없음): 339개
GICS 정보 완전 결측: 10개
GICS 정보 부분 결측: 0개

각 컬럼별 고유값 개수:
  sector: 8개
  industry_group: 15개
  industry: 34개
  sub_industry: 53개

💾 파일 저장
✅ [전체 데이터] 349개 행 저장
   📂 ../output/gics_all.csv
✅ [결측치 제거] 339개 행 저장
   📂 ../output/gic_clean.csv

💡 사용 가이드
• gics_all.csv (349개)
  → 가격 데이터만으로 예측할 때 사용
  → 모든 symbol 포함

• gics_clean.csv (339개)
  → GICS 정보를 feature로 사용할 때
  → 완전한 GICS 정보만 포함
